# 01 · Limpieza de curvas por variedad y camas piloto

**Entrada:** 24 libros originales, cuatro familias y seis ediciones anuales.
**Salida única:** `curvas_variedad_limpio.parquet`, observaciones piloto por siembra/semana de corte.

Los libros contienen historia acumulada anterior a 2021; no se confunde el año de edición con el año de cosecha. Se conserva ese histórico útil. La variable de rendimiento es **flores por planta por semana** del Excel, no el índice interno de WebFlor. No se calculan aquí modelos ni un promedio que use información futura.

## 1. Imports y configuración

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
from datetime import date
import unicodedata
from pathlib import PurePosixPath
from zipfile import ZipFile
from lxml import etree

def texto(valor):
    if pd.isna(valor):
        return ""
    s = unicodedata.normalize("NFKD", str(valor).replace("\xa0", " "))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().upper()

def normalizar(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.replace("[\u0300-\u036f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA}))

def fechas(serie):
    """Conservar calendario de Excel: fechas ya leídas o seriales 1900, no nanosegundos 1970."""
    salida = pd.to_datetime(serie, errors="coerce", format="mixed")
    numeros = pd.to_numeric(serie.astype("string"), errors="coerce")
    serial = numeros.between(20000, 80000)
    salida.loc[serial] = pd.to_datetime(numeros[serial], unit="D", origin="1899-12-30")
    return salida

def semana_operativa(anio, semana):
    """Lunes representativo. Toda etiqueta 53 se convierte en 01 del año siguiente."""
    y = pd.to_numeric(anio, errors="coerce").astype("Int64")
    w = pd.to_numeric(semana, errors="coerce").astype("Int64")
    y = y + w.eq(53).fillna(False).astype(int)
    w = w.mask(w.eq(53), 1)
    etiquetas = y.astype("string") + "-" + w.astype("string")
    conversion = {}
    for etiqueta in etiquetas.dropna().unique():
        try:
            yy, ww = map(int, etiqueta.split("-"))
            conversion[etiqueta] = pd.Timestamp(date.fromisocalendar(yy, ww, 1)) if 1 <= ww <= 52 else pd.NaT
        except (ValueError, OverflowError):
            conversion[etiqueta] = pd.NaT
    return pd.to_datetime(etiquetas.map(conversion))

def leer_tabla_excel(libro, hoja, obligatorias):
    """Detectar el encabezado desplazado; devolver datos y número de fila Excel inicial."""
    inicio = pd.read_excel(libro, sheet_name=hoja, header=None, nrows=12)
    requeridas = {texto(c) for c in obligatorias}
    candidatos = [i for i, fila in inicio.iterrows() if requeridas <= {texto(x) for x in fila}]
    if len(candidatos) != 1:
        raise ValueError(f"Encabezado ambiguo/ausente en {hoja}: {obligatorias}")
    encabezado = candidatos[0]
    d = pd.read_excel(libro, sheet_name=hoja, header=encabezado)
    d.columns = [texto(c) for c in d.columns]
    return d, encabezado + 2

SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

ENTRADA = ROOT / "Datos/Curvas por variedad"
ARCHIVO_SALIDA = SALIDA / "curvas_variedad_limpio.parquet"
archivos = sorted(p for p in ENTRADA.glob("*/*.xlsx") if re.fullmatch(r"20\d{2}",p.parent.name))
display(pd.DataFrame({"archivo":[str(p.relative_to(ENTRADA)) for p in archivos]}))


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


,archivo
0,2021\Estadisticas Clavel 2021.xlsx
1,2021\Estadisticas Green Ball 2021.xlsx
2,2021\Estadisticas Mini Clavel 2021 (Reparado)....
3,"2021\Estadisticas Raffine, Solomio, Otros 2021..."
4,2022\Estadisticas Clavel 2022.xlsx
5,2022\Estadisticas Green Ball 2022.xlsx
6,2022\Estadisticas Mini Clavel 2022.xlsx
7,"2022\Estadisticas Raffine, Solomio, Otros 2022..."
8,2023\Estadisticas Clavel 2023.xlsx
9,2023\Estadisticas Green Ball 2023.xlsx


## 2. Estructura original

Se usa Base de Datos o corte semanal, según el libro, y Camas Piloto para contrastar plantas y fechas cuando la referencia es inequívoca. No se suman reportes y datos.

In [2]:
libro = pd.ExcelFile(archivos[0],engine="calamine")
print("Hojas:",libro.sheet_names)
display(pd.read_excel(libro,sheet_name="Base de Datos",nrows=5))
display(pd.read_excel(libro,sheet_name="Camas Piloto",nrows=5))
libro.close()


Hojas: ['Variedades', 'semana', 'Variedades B29', 'Camas Piloto', 'bloques nuevos', 'Modelo de siembras', 'Base de Datos', 'curvas', 'camas comparacion SG-S90', 'Productividad florxplanta', 'proyecto inc MO', 'Analisis sustratos', 'Laboratorio Sustrato', 'Laboratorio de Sustrato Grafica', 'curvas historicas', 'SUSTRATO', 'xxx', 'FL-PL ACTUAL', 'M2 ACTUAL ', 'Hoja1', 'Informe (2)', 'Informe 1', 'MILENA', 'Sustr']


,Cama Piloto,BLOQUE,CAMA,Semana,Año Corte,Semana Corte,Corte Semanal,Variedad,Color,# Sem Corte,Edad,Indice Prod,Flor/Planta,Fecha de Siembra,Semana de Siembra,Año de Siembra,Sustrato,ancho de cama,Breeder,PRODUCTIVA/NO PRODUCTIVA,Proyecto inc MO,CODIGO SEMSIEMBRA,CODIGO2,SEGUMIENTO SI/NO,VARIEDAD B29?,BLOQUE 29?,error?,BLOQUE-AÑO
0,POMODORO-6-64,6,64,1,2013,201301,115,POMODORO,Rojo,159,79,2.464286,0.082143,2011-07-13,28,2011,S90%,0,Sb Talee,-,0,201128,POMODORO201128,NaN,NO,Otros bloques,ok,6-2013
1,POMODORO-6-66,6,66,1,2013,201301,204,POMODORO,Rojo,159,59,5.953307,0.198444,2011-11-30,48,2011,S90%,0,Sb Talee,-,0,201148,POMODORO201148,NaN,NO,Otros bloques,ok,6-2013
2,CARIBE-6-211,6,211,1,2013,201301,106,CARIBE,Rojo,159,36,3.117647,0.103922,2012-05-12,18,2012,S90%,0,NaN,-,0,201218,CARIBE201218,NaN,NO,Otros bloques,ok,6-2013
3,LION KING-9-74,9,74,1,2013,201301,670,LION KING,Naranja,159,30,19.628906,0.654297,2012-06-19,24,2012,S90%,0,Hilverda Kooij,-,0,201224,LION KING201224,NaN,NO,Otros bloques,ok,9-2013
4,LED-9-100,9,100,1,2013,201301,72,LED,Crema,159,99,2.109375,0.070312,2011-02-26,8,2011,S90%,0,NaN,-,0,20118,LED20118,NaN,NO,Otros bloques,ok,9-2013


,Referencia,Color,Variedad,No. Bloque,No. Cama,Fecha de Siembra,No. Plantas,Sem Siembra,Año Siembra,Sem Siembra.1,#Sem,Inicio Prod.,Densidad de Siembra,Camas del Lote,Plantas del Lote,Sustrato,ancho de cama,Conc,proyecto mo,CODIGO,SEGUIMIENTO SI/NO
0,DILETTA CREMA-14-18,Crema,DILETTA CREMA,14,18,2016-01-22,1872,3,2016,201603,319,20,30,NaN,NaN,S90%,NaN,NaN,NaN,DILETTA CREMA20163,NO SEGUIMIENTO
1,ZEPPELIN-14-21,Cereza,ZEPPELIN,14,21,2016-11-19,1620,46,2016,201646,362,20,30,NaN,NaN,S90%,64.0,NaN,NaN,NaN,NaN
2,NOVIA-14-26,Durazno,NOVIA,14,26,2016-09-10,1860,36,2016,201636,352,20,30,NaN,NaN,S90%,NaN,NaN,NaN,NaN,NaN
3,BIZET-14-36,Cereza,BIZET,14,36,2016-01-12,1782,2,2016,201602,318,20,30,NaN,NaN,S90%,NaN,NaN,NaN,NaN,NaN
4,GLADYS-14-46,Amarillo,GLADYS,14,46,2016-09-23,1842,38,2016,201638,354,20,30,NaN,NaN,S90%,NaN,NaN,NaN,NaN,NaN


## 3. Extraer observaciones y complementar únicamente llaves seguras

No se fuerza el cruce de camas. Si la pareja referencia/fecha tiene varios números de plantas, queda sin dato piloto utilizable. Green Ball a veces sólo tiene año/semana de siembra: se conserva esa precisión semanal y se marca, en vez de inventar un día real.

In [3]:
alias = {"CAMA PILOTO":"referencia","CODIGO":"referencia","BLOQUE":"bloque","CAMA":"cama",
         "ANO CORTE":"anio_corte_fuente","SEMANA CORTE":"codigo_semana_corte",
         "CORTE SEMANAL":"tallos_piloto","VARIEDAD":"variedad","COLOR":"color_piloto",
         "EDAD":"edad_semanas","FLOR/PLANTA":"flores_por_planta","FLORES X PLANTA":"flores_por_planta",
         "FECHA DE SIEMBRA":"fecha_siembra","SEMANA DE SIEMBRA":"semana_siembra_fuente",
         "ANO DE SIEMBRA":"anio_siembra_fuente","ANO SIEMBRA":"anio_siembra_fuente",
         "SEMANA SIEMBRA":"semana_siembra_fuente","PRODUCTO":"producto"}
partes, controles = [], []
for archivo in archivos:
    nombre = texto(archivo.name)
    familia = "MINICARNATION" if "MINI" in nombre else "GREEN BALL" if "GREEN" in nombre else "MIXTA" if ("RAFFINE" in nombre or "SOLOMIO" in nombre) else "CARNATION"
    libro = pd.ExcelFile(archivo,engine="calamine")
    hojas = [h for h in libro.sheet_names if texto(h) in ["BASE DE DATOS","CORTE SEMANAL"]]
    assert len(hojas)==1,(archivo.name,hojas)
    df, inicio = leer_tabla_excel(libro,hojas[0],["VARIEDAD","EDAD","CORTE SEMANAL"])
    df = df[[c for c in df if c in alias]].rename(columns=alias)
    assert not df.columns.duplicated().any(),archivo.name
    df = df.reindex(columns=sorted(set(alias.values())))
    df["fila_excel"] = np.arange(inicio,inicio+len(df))
    df = df.loc[df[["referencia","variedad","tallos_piloto"]].notna().any(axis=1)].copy()
    for c in ["referencia","bloque","cama","variedad","color_piloto","producto"]:
        df[c] = normalizar(df[c])
    if familia!="MIXTA": df["producto"] = familia
    for c in ["anio_corte_fuente","codigo_semana_corte","edad_semanas","tallos_piloto",
              "flores_por_planta","anio_siembra_fuente","semana_siembra_fuente"]:
        df[c] = pd.to_numeric(df[c],errors="coerce")
    df["semana_corte_fuente"] = df.codigo_semana_corte % 100
    df["semana_inicio"] = semana_operativa(df.anio_corte_fuente,df.semana_corte_fuente)
    df["fecha_siembra"] = fechas(df.fecha_siembra).dt.normalize()
    # Green Ball usa en algunos libros AAAASS en SEMANA SIEMBRA.
    df["semana_siembra_fuente"] = df.semana_siembra_fuente % 100
    siembra_sem = semana_operativa(df.anio_siembra_fuente,df.semana_siembra_fuente)
    df["precision_siembra"] = np.where(df.fecha_siembra.notna(),"dia","semana")
    df["fecha_siembra"] = df.fecha_siembra.fillna(siembra_sem)
    hojas_piloto = [h for h in libro.sheet_names if texto(h)=="CAMAS PILOTO"]
    df["plantas_piloto"] = np.nan
    if hojas_piloto:
        pil, _ = leer_tabla_excel(libro,hojas_piloto[0],["REFERENCIA","VARIEDAD","NO. PLANTAS"])
        pil = pil.rename(columns={"REFERENCIA":"referencia","FECHA DE SIEMBRA":"fecha_siembra","NO. PLANTAS":"plantas"})
        pil["referencia"] = normalizar(pil.referencia)
        pil["fecha_siembra"] = fechas(pil.fecha_siembra).dt.normalize()
        pil["plantas"] = pd.to_numeric(pil.plantas,errors="coerce")
        pk = ["referencia","fecha_siembra"]
        unicas = pil.groupby(pk,dropna=False).plantas.nunique().eq(1)
        validas = unicas[unicas].reset_index()[pk]
        pil = pil.merge(validas,on=pk,validate="many_to_one").drop_duplicates(pk)
        unido = df.drop(columns="plantas_piloto").merge(pil[pk+["plantas"]],on=pk,how="left",validate="many_to_one")
        assert len(unido)==len(df)
        df = unido.rename(columns={"plantas":"plantas_piloto"})
    df["rendimiento_recalculado"] = df.tallos_piloto / df.plantas_piloto.where(df.plantas_piloto.gt(0))
    df["familia_archivo"],df["edicion_anio"] = familia,int(archivo.parent.name)
    df["archivo_origen"] = str(archivo.relative_to(ROOT)).replace("\\","/")
    df["hoja_origen"] = hojas[0]
    partes.append(df)
    controles.append({"archivo":archivo.name,"familia":familia,"filas":len(df),"con_plantas_piloto":int(df.plantas_piloto.notna().sum())})
    libro.close()
    print(archivo.parent.name,archivo.name,len(df),flush=True)
curvas = pd.concat(partes,ignore_index=True)
del partes, df
display(pd.DataFrame(controles))


2021 Estadisticas Clavel 2021.xlsx 96962


2021 Estadisticas Green Ball 2021.xlsx 3127


2021 Estadisticas Mini Clavel 2021 (Reparado).xlsx 129971


2021 Estadisticas Raffine, Solomio, Otros 2021.xlsx 19683


2022 Estadisticas Clavel 2022.xlsx 109154


...

2026 Estadisticas Clavel 2026.xlsx 170140


2026 Estadisticas Green Ball 2026.xlsx 6465


2026 Estadisticas Miniclavel 2026.xlsx 244450


2026 Estadisticas Raffine_Solomio_Otros 2026.xlsx 109195


,archivo,familia,filas,con_plantas_piloto
0,Estadisticas Clavel 2021.xlsx,CARNATION,96962,92315
1,Estadisticas Green Ball 2021.xlsx,GREEN BALL,3127,126
2,Estadisticas Mini Clavel 2021 (Reparado).xlsx,MINICARNATION,129971,125044
3,"Estadisticas Raffine, Solomio, Otros 2021.xlsx",MIXTA,19683,18688
4,Estadisticas Clavel 2022.xlsx,CARNATION,109154,103698
5,Estadisticas Green Ball 2022.xlsx,GREEN BALL,3369,126
6,Estadisticas Mini Clavel 2022.xlsx,MINICARNATION,157739,152210
7,"Estadisticas Raffine, Solomio, Otros 2022.xlsx",MIXTA,33814,32635
8,Estadisticas Clavel 2023.xlsx,CARNATION,122933,117448
9,Estadisticas Green Ball 2023.xlsx,GREEN BALL,4162,132


## 4. Desduplicación del histórico acumulativo

Llave de evento: familia, referencia, fecha/precisión de siembra y etiqueta de corte original. Se elige la primera edición disponible de ese evento, no la revisión futura. Los conflictos dentro de esa primera edición se marcan y no alimentan el potencial. Las revisiones posteriores quedan contabilizadas; los Excel originales siguen intactos.

In [4]:
evento = ["familia_archivo","referencia","fecha_siembra","anio_corte_fuente","semana_corte_fuente"]
n_entrada = len(curvas)
curvas["primera_edicion"] = curvas.groupby(evento,dropna=False).edicion_anio.transform("min")
curvas["repeticiones_fuente"] = curvas.groupby(evento,dropna=False).referencia.transform("size")
primera = curvas.loc[curvas.edicion_anio.eq(curvas.primera_edicion)].copy()
conflictos = primera.groupby(evento,dropna=False)[["variedad","edad_semanas","flores_por_planta","tallos_piloto"]].nunique().gt(1).any(axis=1).rename("conflicto_primera_edicion")
curvas = primera.sort_values(["archivo_origen","fila_excel"]).drop_duplicates(evento)
curvas = curvas.merge(conflictos,on=evento,validate="one_to_one")
curvas["observacion_valida"] = (
    curvas[["variedad","producto","referencia","fecha_siembra","semana_inicio"]].notna().all(axis=1)
    & curvas.edad_semanas.ge(0) & curvas.edad_semanas.eq(np.floor(curvas.edad_semanas))
    & curvas.flores_por_planta.ge(0) & ~curvas.conflicto_primera_edicion
    & (curvas.codigo_semana_corte//100).eq(curvas.anio_corte_fuente))
# Sólo un supuesto conservador, NO fecha real de publicación del libro.
fecha_edicion = pd.to_datetime((curvas.edicion_anio+1).astype("string")+"-01-01")
curvas["disponible_supuesto"] = pd.concat([fecha_edicion,curvas.semana_inicio+pd.Timedelta(days=7)],axis=1).max(axis=1)
curvas["disponibilidad_confirmada"] = False
assert not curvas.duplicated(evento).any()
print("Filas acumuladas:",n_entrada,"Eventos únicos:",len(curvas),"Admisibles:",int(curvas.observacion_valida.sum()))
display(curvas.groupby("familia_archivo").agg(eventos=("referencia","size"),desde=("semana_inicio","min"),
                  hasta=("semana_inicio","max"),validas=("observacion_valida","sum")))
display(curvas[["edad_semanas","flores_por_planta","plantas_piloto","fecha_siembra"]].isna().sum().to_frame("faltantes"))


Filas acumuladas: 2376366 Eventos únicos: 549870 Admisibles: 502501


,eventos,desde,hasta,validas
familia_archivo,,,,
CARNATION,172648,2012-12-31,2026-07-13,156654
GREEN BALL,7932,2014-06-09,2026-07-13,6332
MINICARNATION,256020,2013-12-30,2026-07-13,230898
MIXTA,113270,2017-05-01,2026-07-06,108617


,faltantes
edad_semanas,19018
flores_por_planta,16787
plantas_piloto,25241
fecha_siembra,15325


## 5. Exportación única y resumen

No se genera otro Parquet de pilotos ni CSV por libro. Los conflictos y las observaciones no admisibles permanecen marcados dentro de esta salida. Las curvas empíricas agregadas se calculan en memoria durante la construcción analítica.

In [5]:
curvas.to_parquet(ARCHIVO_SALIDA,index=False)
print("EXPORTADO:",ARCHIVO_SALIDA,curvas.shape)
display(curvas.head())


EXPORTADO: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\curvas_variedad_limpio.parquet (549870, 30)


,anio_corte_fuente,anio_siembra_fuente,bloque,cama,codigo_semana_corte,color_piloto,edad_semanas,fecha_siembra,flores_por_planta,producto,referencia,semana_siembra_fuente,tallos_piloto,variedad,fila_excel,semana_corte_fuente,semana_inicio,precision_siembra,plantas_piloto,rendimiento_recalculado,familia_archivo,edicion_anio,archivo_origen,hoja_origen,primera_edicion,repeticiones_fuente,conflicto_primera_edicion,observacion_valida,disponible_supuesto,disponibilidad_confirmada
0,2013.0,2011.0,6.0,64,201301.0,ROJO,79.0,2011-07-13,0.082143,CARNATION,POMODORO-6-64,28.0,115.0,POMODORO,2,1.0,2012-12-31,dia,1400.0,0.082143,CARNATION,2021,Datos/Curvas por variedad/2021/Estadisticas Cl...,Base de Datos,2021,6,False,True,2022-01-01,False
1,2013.0,2011.0,6.0,66,201301.0,ROJO,59.0,2011-11-30,0.198444,CARNATION,POMODORO-6-66,48.0,204.0,POMODORO,3,1.0,2012-12-31,dia,1028.0,0.198444,CARNATION,2021,Datos/Curvas por variedad/2021/Estadisticas Cl...,Base de Datos,2021,6,False,True,2022-01-01,False
2,2013.0,2012.0,6.0,211,201301.0,ROJO,36.0,2012-05-12,0.103922,CARNATION,CARIBE-6-211,18.0,106.0,CARIBE,4,1.0,2012-12-31,dia,1020.0,0.103922,CARNATION,2021,Datos/Curvas por variedad/2021/Estadisticas Cl...,Base de Datos,2021,6,False,True,2022-01-01,False
3,2013.0,2012.0,9.0,74,201301.0,NARANJA,30.0,2012-06-19,0.654297,CARNATION,LION KING-9-74,24.0,670.0,LION KING,5,1.0,2012-12-31,dia,1024.0,0.654297,CARNATION,2021,Datos/Curvas por variedad/2021/Estadisticas Cl...,Base de Datos,2021,6,False,True,2022-01-01,False
4,2013.0,2011.0,9.0,100,201301.0,CREMA,99.0,2011-02-26,0.070312,CARNATION,LED-9-100,8.0,72.0,LED,6,1.0,2012-12-31,dia,1024.0,0.070312,CARNATION,2021,Datos/Curvas por variedad/2021/Estadisticas Cl...,Base de Datos,2021,6,False,True,2022-01-01,False
